# Помесячная статистика портфеля

Ноутбук:
- находит все даты по колонкам `OD_{дата}`;
- использует для каждого месяца `OD`, `НИ`, `ПФН`, `рестра`, `Обеспеченность`;
- оставляет разбивку **Сегмент → Тип операции (Актив/УО)**;
- распределяет OD по 7 критериям;
- создает отдельный DataFrame на каждую дату в `summary_by_date`;
- сохраняет каждую дату на отдельный лист Excel.

> Если исходник в `.numbers`, сначала экспортируйте его в `.xlsx`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)


## 1. Настройки

In [ ]:
INPUT_FILE = Path("portfolio_monthly.xlsx")
OUTPUT_FILE = Path("Статистика_помесячно.xlsx")

SHEET_NAME = 0

COL_SEGMENT = "Сегмент"
COL_OPERATION = "Тип операции"

# Если OD в тыс. BYN, на выходе будет млн BYN
DIVIDE_OD_BY_1000 = True


## 2. Вспомогательные функции

In [ ]:
def prepare_flag(series):
    s = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(",", ".", regex=False)
    )

    mapping = {
        "1": 1, "1.0": 1, "да": 1, "yes": 1, "true": 1,
        "0": 0, "0.0": 0, "нет": 0, "no": 0, "false": 0,
        "nan": 0, "none": 0, "": 0,
    }

    result = s.map(mapping)
    numeric = pd.to_numeric(s, errors="coerce")
    return result.fillna(numeric)


def prepare_number(series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce").fillna(0)

    s = (
        series.astype(str)
        .str.replace("\xa0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    return pd.to_numeric(s, errors="coerce").fillna(0)


def find_month_column(df, prefix, date):
    target = f"{prefix}_{date}".strip().lower().replace("ё", "е")

    for col in df.columns:
        col_norm = str(col).strip().lower().replace("ё", "е")
        if col_norm == target:
            return col

    return None


## 3. Чтение файла

In [ ]:
df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)

print(f"Строк: {len(df):,}")
print(f"Колонок: {len(df.columns):,}")
display(df.head())


## 4. Находим все даты по колонкам OD_*

In [ ]:
dates = []

for col in df.columns:
    col_str = str(col).strip()

    if col_str.lower().startswith("od_"):
        dates.append(col_str[3:])

dates = list(dict.fromkeys(dates))

print("Найденные даты:")
for date in dates:
    print(date)

print(f"\nВсего дат: {len(dates)}")


## 5. Критерии

In [ ]:
C1 = "1. НИ=0; ПФН=0; Рестра=0; обеспеченный"
C2 = "2. НИ=0; ПФН=0; Рестра=0; необеспеченный"
C3 = "3. НИ=1; ПФН=0; Рестра=0; обеспеченный"
C4 = "4. НИ=1; ПФН=0; Рестра=0; необеспеченный"
C5 = "5. ПФН=1; обеспеченный"
C6 = "6. ПФН=1; необеспеченный"
C7 = "7. Рестра=1"

criteria_order = [C1, C2, C3, C4, C5, C6, C7]


## 6. Создаем отдельный DataFrame для каждой даты

Результаты будут доступны так:

```python
summary_by_date["31.01.2026"]
summary_by_date["28.02.2026"]
```


In [ ]:
summary_by_date = {}
raw_by_date = {}
skipped_dates = {}

for date in dates:

    col_od = find_month_column(df, "OD", date)
    col_ni = find_month_column(df, "НИ", date)
    col_pfn = find_month_column(df, "ПФН", date)
    col_restra = find_month_column(df, "рестра", date)
    col_collateral = find_month_column(df, "Обеспеченность", date)

    month_cols = {
        "OD": col_od,
        "НИ": col_ni,
        "ПФН": col_pfn,
        "рестра": col_restra,
        "Обеспеченность": col_collateral,
    }

    missing = [
        name
        for name, col in month_cols.items()
        if col is None
    ]

    if missing:
        skipped_dates[date] = missing
        print(f"Пропуск {date}: нет колонок {missing}")
        continue

    tmp = pd.DataFrame({
        "Сегмент": df[COL_SEGMENT],
        "Тип операции": df[COL_OPERATION],
        "Дата": date,
    })

    tmp["OD"] = prepare_number(df[col_od])

    if DIVIDE_OD_BY_1000:
        tmp["OD"] = tmp["OD"] / 1000

    tmp["НИ"] = prepare_flag(df[col_ni])
    tmp["ПФН"] = prepare_flag(df[col_pfn])
    tmp["Рестра"] = prepare_flag(df[col_restra])

    collateral = (
        df[col_collateral]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace("ё", "е")
    )

    tmp["Необеспеченный"] = collateral.str.contains(
        "необеспеч",
        na=False
    )

    tmp["Обеспеченный"] = ~tmp["Необеспеченный"]

    conditions = [
        tmp["Рестра"].eq(1),

        (
            tmp["Рестра"].eq(0)
            & tmp["ПФН"].eq(1)
            & tmp["Обеспеченный"]
        ),

        (
            tmp["Рестра"].eq(0)
            & tmp["ПФН"].eq(1)
            & tmp["Необеспеченный"]
        ),

        (
            tmp["Рестра"].eq(0)
            & tmp["ПФН"].eq(0)
            & tmp["НИ"].eq(1)
            & tmp["Обеспеченный"]
        ),

        (
            tmp["Рестра"].eq(0)
            & tmp["ПФН"].eq(0)
            & tmp["НИ"].eq(1)
            & tmp["Необеспеченный"]
        ),

        (
            tmp["Рестра"].eq(0)
            & tmp["ПФН"].eq(0)
            & tmp["НИ"].eq(0)
            & tmp["Обеспеченный"]
        ),

        (
            tmp["Рестра"].eq(0)
            & tmp["ПФН"].eq(0)
            & tmp["НИ"].eq(0)
            & tmp["Необеспеченный"]
        ),
    ]

    choices = [C7, C5, C6, C3, C4, C1, C2]

    tmp["Критерий"] = np.select(
        conditions,
        choices,
        default="НЕ РАСПРЕДЕЛЕНО"
    )

    operation_norm = (
        tmp["Тип операции"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    tmp = tmp[
        operation_norm.isin(["актив", "уо"])
    ].copy()

    raw_by_date[date] = tmp

    summary_date = (
        tmp[
            tmp["Критерий"] != "НЕ РАСПРЕДЕЛЕНО"
        ]
        .pivot_table(
            index=[
                "Сегмент",
                "Тип операции",
            ],
            columns="Критерий",
            values="OD",
            aggfunc="sum",
            fill_value=0,
        )
        .reindex(
            columns=criteria_order,
            fill_value=0
        )
        .reset_index()
    )

    summary_date["ИТОГО"] = (
        summary_date[criteria_order]
        .sum(axis=1)
    )

    summary_by_date[date] = summary_date


print(
    f"Сформировано DataFrame: "
    f"{len(summary_by_date)}"
)


## 7. Список созданных DataFrame

In [ ]:
list(summary_by_date.keys())


## 8. Вывести все даты

In [ ]:
for date, df_date in summary_by_date.items():
    print(f"\n===== {date} =====")
    display(df_date)


## 9. Получить конкретную дату

Например:

```python
df_jan = summary_by_date["31.01.2026"]
df_jan
```


In [ ]:
# Автоматически покажем первую найденную дату
if summary_by_date:
    first_date = next(iter(summary_by_date))
    print("Первая дата:", first_date)
    display(summary_by_date[first_date])


## 10. Сохранить каждый месяц на отдельный лист Excel

In [ ]:
with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="xlsxwriter"
) as writer:

    for date, df_date in summary_by_date.items():

        sheet_name = str(date)

        # Недопустимые символы Excel
        for ch in ["/", "\\", ":", "*", "?", "[", "]"]:
            sheet_name = sheet_name.replace(ch, ".")

        sheet_name = sheet_name[:31]

        df_date.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

        ws = writer.sheets[sheet_name]

        ws.freeze_panes(1, 2)

        ws.set_column(0, 0, 18)
        ws.set_column(1, 1, 15)
        ws.set_column(
            2,
            len(df_date.columns) - 1,
            22
        )


print(f"Готово: {OUTPUT_FILE.resolve()}")
